# C3 — Why: is the limit the number of measurements, or the number of positions? (Stages 3c–3e)

**Question.** PSD95-PDZ3 is learnable and GRB2-SH3 is not. PSD95 has twice the measurements
*and* 1.5× the positions. Which one matters — and could more GRB2-SH3 mutagenesis ever fix it?

**What it does**, each step answering the question the previous one raised:
1. **Matched budget** (3c): cut PSD95-PDZ3 to GRB2-SH3's exact budget (56 positions, 619
   measurements), 20 draws. Does it collapse to GRB2's level?
2. **Diagnostic grid** (3c): the same full data at 5 vs 7 folds and 40 vs 60 epochs.
3. **Learning curve** (3c): both domains across six budgets.
4. **Three sweeps** (3d): A varies only the fold count; B thins measurements but keeps all 84
   positions; C removes positions at matched measurement budgets. If B and C disagree, *how* the
   data is thin matters.
5. **Position sweep** (3e): every position count from 60 to 84, ten random subsets each, with
   shuffled partitions. Where does predictability appear, and is it *which* positions or *how many*?

**Two partition modes, run side by side.** Re-checking the original code showed that Stages
3b–3d split positions into folds with scikit-learn's `GroupKFold`, which assigns groups
deterministically and does not shuffle by default. Every "repeat" therefore reused the same
partition; only the inner validation split and the model seeds changed.
- `original` — the unshuffled procedure, reproducing the recorded numbers exactly.
- `corrected` — positions shuffled with the seed before assignment (the Stage 3e design), so each
  repeat is a genuinely different partition.
Set `MODES` below to run one or both. The results table lists them side by side.

Step 5 already used shuffled partitions in the original run, so it runs once.

**Inputs:** A2 features, C2 repeats. **Runtime:** 30–45 min (both modes), T4 GPU; each step
resumes from its own file. **Outputs:** `matched_budget.csv`, `diagnostic_grid.csv`,
`learning_curve.csv`, `threshold_sweeps.csv`, `position_sweep.csv`, `position_leverage.csv`,
`RESULTS_C3.md`. Explanation: `docs/C3_binding_data_sufficiency.md`.

In [ ]:
# ── setup (identical in every notebook) ─────────────────────────────────────────
ROOT   = '/content/drive/MyDrive/sh3_repro'   # the Drive folder from UPLOAD_GUIDE.md
RESUME = True                                  # reuse finished steps after a disconnect

import os, sys, json, time, importlib
from google.colab import drive; drive.mount('/content/drive')
sys.path.insert(0, f'{ROOT}/code')
import sh3_common as C; importlib.reload(C)
import numpy as np, pandas as pd
R = C.Run(ROOT, 'C3', 'Binding data sufficiency', resume=RESUME)
if C.DEV == 'cpu':
    print('WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU.')
print(C.env_info())

In [ ]:
MODES = ['original', 'corrected']   # either or both

DD = pd.read_parquet(R.res('A2', 'features_ddpca.parquet'))
EMB = C.load_embeddings(R.res('A2', 'esm_domain_embeddings.npz'))
Xd, DB = C.build_X(DD, EMB)
B = DB[DB.ddG_bind.notna() & DB.binding_reliable.eq(True)].copy()
XB = Xd[B.index.values].astype(np.float32); B = B.reset_index(drop=True)
print(B.groupby('domain_key').agg(measurements=('pos', 'size'), positions=('pos', 'nunique')).to_string())

DOM = 'PSD95-PDZ3'; D = B[B.domain_key == DOM]; ALLPOS = np.sort(D.pos.unique())
C2S = pd.read_csv(R.res('C2', 'within_domain_summary.csv'))
def cached(name, fn):
    if R.done(name): return pd.read_csv(R.out(name))
    t0 = time.time(); df = pd.DataFrame(fn()); df.to_csv(R.out(name), index=False)
    print(f'{name}: {len(df)} runs, {(time.time() - t0) / 60:.1f} min'); return df

## 1. Matched budget: PSD95-PDZ3 cut to GRB2-SH3's data

In [ ]:
MB = cached('matched_budget.csv', lambda: [r for m in MODES for s in range(20)
            if (r := C.eval_budget(B, XB, DOM, 56, 619, seed=s, n_splits=7, epochs=60, mode=m))])
MBS = MB.groupby('mode').rho.agg(median='median', ci_lo=lambda x: np.percentile(x, 2.5),
                                 ci_hi=lambda x: np.percentile(x, 97.5), draws='size').reset_index()
MBS = MBS.merge(C2S[C2S.domain == 'GRB2-SH3'][['mode', 'repeats_median']].rename(columns={'repeats_median': 'grb2_median'}), on='mode')
MBS['covers_grb2'] = (MBS.ci_lo <= MBS.grb2_median) & (MBS.grb2_median <= MBS.ci_hi); print(MBS.round(3).to_string(index=False))

## 2. Diagnostic grid and 3. learning curve

In [ ]:
DG = cached('diagnostic_grid.csv', lambda: [dict(r) for m in MODES for ns in (7, 5) for ep in (60, 40) for s in range(6)
            if (r := C.eval_budget(B, XB, DOM, D.pos.nunique(), len(D), seed=s, n_splits=ns, model_seeds=1, epochs=ep, mode=m))])
BUDGETS = [(25, 200), (35, 350), (45, 500), (56, 619), (70, 850), (84, 1221)]
def lc_runs():
    for m in MODES:
        for dom in ['PSD95-PDZ3', 'GRB2-SH3']:
            d = B[B.domain_key == dom]
            for npos, nrow in BUDGETS:
                if npos > d.pos.nunique() or nrow > len(d): continue
                for s in range(6):
                    r = C.eval_budget(B, XB, dom, npos, nrow, seed=s, n_splits=7, epochs=60, mode=m)
                    if r: yield r
LC = cached('learning_curve.csv', lambda: list(lc_runs()))
DGS = DG.groupby(['mode', 'n_splits', 'epochs']).rho.median().rename('median_rho').reset_index()
LCS = LC.groupby(['mode', 'domain', 'n_pos', 'n']).rho.agg(['median', 'min', 'max', 'size']).reset_index()
print(DGS.round(3).to_string(index=False)); print(LCS.round(3).to_string(index=False))

## 4. Three sweeps: fold count (A), thinned rows at all positions (B), fewer positions (C)

In [ ]:
def sweeps():
    for m in MODES:
        for k in (3, 4, 5, 6, 7, 8, 10):
            for s in range(6):
                r = C.run_cv_3d(D, XB, n_splits=k, seed=s, mode=m)
                if r: yield {**r, 'sweep': 'A_folds'}
        for budget in (700, 800, 900, 1000, 1100, 1221):
            for s in range(6):
                rng = np.random.default_rng(1000 + s)
                sub = D.sample(min(budget, len(D)), random_state=1000 + s)
                r = C.run_cv_3d(sub, XB, n_splits=7, seed=s, mode=m)
                if r: yield {**r, 'sweep': 'B_thin'}
                npos = max(8, int(round(84 * budget / len(D))))
                pos = rng.permutation(D.pos.unique())[:npos]
                sub = D[D.pos.isin(pos)]
                if len(sub) > budget: sub = sub.sample(budget, random_state=1000 + s)
                r = C.run_cv_3d(sub, XB, n_splits=7, seed=s, mode=m)
                if r: yield {**r, 'sweep': 'C_fewpos'}
SW = cached('threshold_sweeps.csv', lambda: list(sweeps()))
SWS = SW.groupby(['mode', 'sweep']).rho.agg(median='median', runs='size').reset_index(); print(SWS.round(3).to_string(index=False))
GAP = {}
for m, g in SW.groupby('mode'):
    piv = g.assign(bin=g.train_rows.round(-2)).pivot_table(index='bin', columns='sweep', values='rho', aggfunc='median').dropna()
    GAP[m] = float((piv.max(axis=1) - piv.min(axis=1)).max()) if len(piv) >= 3 else np.nan
from scipy.optimize import curve_fit
logistic = lambda x, L, k, x0: L / (1 + np.exp(-k * (x - x0)))
FITS = []
for m, g in SW.groupby('mode'):
    for label, sel in [('all sweeps', g), ('A only', g[g.sweep == 'A_folds'])]:
        q = sel.groupby(sel.train_rows.round(-1)).rho.median()
        try:
            p, _ = curve_fit(logistic, np.log10(q.index.values.astype(float)), q.values, p0=[0.4, 20, 3.0],
                             bounds=([0.05, 1, 2.5], [1.0, 300, 3.3]), maxfev=20000)
            FITS.append(dict(mode=m, fit=label, plateau=p[0], steepness=p[1], midpoint_rows=10 ** p[2]))
        except Exception as e:
            FITS.append(dict(mode=m, fit=label, note=type(e).__name__))
print('max disagreement between sweeps at a shared training size:', GAP)

## 5. Position sweep: 60 to 84 positions, ten shuffled subsets each

In [ ]:
from scipy.stats import mannwhitneyu
COUNTS = sorted(set(range(60, 85, 2)) | {84})
def pos_runs():
    for npos in COUNTS:
        for s in range(10):
            keep = np.random.default_rng(5000 + s).permutation(ALLPOS)[:npos]
            sub = D[D.pos.isin(keep)].reset_index(); Xs = XB[sub['index'].values]; sub = sub.drop(columns=['index'])
            r = C.run_cv_3e(sub, Xs, n_splits=7, seed=s)
            if r: yield {**r, 'target_pos': npos}
PS = cached('position_sweep.csv', lambda: list(pos_runs()))
PSS = PS.groupby('target_pos').rho.agg(['median', 'min', 'max', 'size']).reset_index()
base = PS[PS.target_pos == COUNTS[0]].rho.values; cross = None; mw = []
for n in COUNTS[1:]:
    v = PS[PS.target_pos == n].rho.values; p = mannwhitneyu(v, base, alternative='greater').pvalue
    mw.append(dict(positions=n, median=np.median(v), p_vs_60=p))
    if p < 0.05 and cross is None: cross = n
hi, lo = PS[PS.rho > 0.25].target_pos, PS[PS.rho < 0.15].target_pos
overlap = sorted(set(hi) & set(lo))
from collections import Counter
good, bad = Counter(), Counter()
for _, r in PS.iterrows():
    keep = set(np.random.default_rng(5000 + int(r.seed)).permutation(ALLPOS)[:int(r.target_pos)])
    tgt = good if r.rho > 0.25 else (bad if r.rho < 0.15 else None)
    if tgt is not None:
        for p_ in keep: tgt[p_] += 1
LEV = pd.DataFrame([dict(pos=int(p_), in_success=good.get(p_, 0), in_failure=bad.get(p_, 0),
                         enrichment=(good.get(p_, 0) + 1) / (bad.get(p_, 0) + 1)) for p_ in ALLPOS
                    if good.get(p_, 0) + bad.get(p_, 0) >= 8]).sort_values('enrichment', ascending=False)
LEV.to_csv(R.out('position_leverage.csv'), index=False)
GRB2_POS = int(B[B.domain_key == 'GRB2-SH3'].pos.nunique())
print(PSS.round(3).to_string(index=False)); print(f'crossing at {cross} positions; GRB2-SH3 has {GRB2_POS}')
print(f'position enrichment ranges {LEV.enrichment.min():.2f}-{LEV.enrichment.max():.2f}: no position stands out')

## Results

In [ ]:
for _, r in MBS.iterrows():
    R.metric(f'{r["mode"]}.matched_median_rho', r['median']); R.metric(f'{r["mode"]}.matched_ci_lo', r.ci_lo)
    R.metric(f'{r["mode"]}.matched_ci_hi', r.ci_hi); R.metric(f'{r["mode"]}.matched_draws', r.draws)
    R.metric(f'{r["mode"]}.matched_covers_grb2', r.covers_grb2)
for _, r in DGS.iterrows(): R.metric(f'{r["mode"]}.diag_k{r.n_splits}_ep{r.epochs}_median', r.median_rho)
for _, r in LCS.iterrows(): R.metric(f'{r["mode"]}.lc_{r.domain}_{r.n_pos}pos_median', r['median'])
for _, r in SWS.iterrows():
    R.metric(f'{r["mode"]}.sweep_{r.sweep}_median', r['median']); R.metric(f'{r["mode"]}.sweep_{r.sweep}_runs', r.runs)
for m in SW['mode'].unique():
    R.metric(f'{m}.sweep_max_gap', GAP[m]); R.metric(f'{m}.sweep_runs', int((SW['mode'] == m).sum()))
for _, r in PSS.iterrows(): R.metric(f'position_sweep_{int(r.target_pos)}_median', r['median'])
for k, v in [('position_sweep_runs', len(PS)), ('crossing_positions', cross), ('grb2_positions', GRB2_POS),
             ('success_failure_share_counts', len(overlap)), ('position_enrichment_max', LEV.enrichment.max())]:
    R.metric(k, v)
R.section('1. Matched budget', 'PSD95-PDZ3 at 56 positions / 619 measurements, 20 draws, against GRB2-SH3 (C2).', MBS, '{:.3f}')
R.section('2. Diagnostic grid (PSD95-PDZ3, full data)', 'A result that swings with the fold count is fragile.', DGS, '{:.3f}')
R.section('3. Learning curve', 'Superseded as a model by steps 4–5: a log-linear fit through a step is the wrong shape.', LCS, '{:.3f}')
R.section('4. Three sweeps', f'Maximum disagreement between sweeps at a shared training size: {GAP}. B keeps all positions and '
          'holds its accuracy; C removes positions at the same row budget and loses it.', SWS, '{:.3f}')
R.section('4b. Logistic fits (superseded by step 5)', 'Kept for the record; the apparent sharp step came from the unshuffled '
          'partition.', pd.DataFrame(FITS), '{:.3g}')
R.section('5. Position sweep (shuffled partitions)', f'First count significantly above 60 positions (Mann–Whitney, p < 0.05): '
          f'**{cross}**. GRB2-SH3 has {GRB2_POS} positions in total.', PSS.merge(pd.DataFrame(mw).rename(columns={'positions': 'target_pos'}),
          on='target_pos', how='left'), '{:.3g}')
R.section('5b. Is it which positions, or how many?', f"Counts where both successes (ρ > 0.25) and failures (ρ < 0.15) occur: "
          f"{overlap}. The original notebook read any overlap as 'leverage'. The enrichment table says otherwise: every position "
          f"scores {LEV.enrichment.min():.2f}–{LEV.enrichment.max():.2f}, i.e. the base rate of successes to failures. No position "
          f"carries the signal; the controlling variable is the **count**.")
R.finish([f"Matched budget: PSD95-PDZ3 cut to GRB2-SH3's data falls to median ρ "
          + ', '.join(f"{r['median']:+.3f} ({r['mode']})" for _, r in MBS.iterrows()) + '.',
          'Keeping all positions while cutting rows costs almost nothing; cutting positions at the same row budget collapses it: '
          + ', '.join(f"{m}: B {SWS[(SWS['mode']==m)&(SWS.sweep=='B_thin')]['median'].iat[0]:+.3f} vs C "
                      f"{SWS[(SWS['mode']==m)&(SWS.sweep=='C_fewpos')]['median'].iat[0]:+.3f}" for m in SWS['mode'].unique()) + '.',
          f"Predictability appears at about {cross} positions (solid by 82); GRB2-SH3 has {GRB2_POS}.",
          'Single-domain ddPCA cannot supply enough positions for an SH3 domain (56–60 residues): the fix is more domains, not '
          'more mutations. The number is calibrated on one domain — quote "of the order of 80", not a precise cut.'])